# Практикум 3. Multiclass и multilabel

Сопровождает интерактивную лабораторию №3 («Multiclass и multilabel»). Тема курса: модели многоклассовой и многометочной классификации

Ноутбук самодостаточен: первая ячейка создаёт файл `common.py` с учебным корпусом и вспомогательными функциями. Запускайте ячейки сверху вниз. Корпус учебный (шаблонный), для реальной задачи подставьте свой CSV со столбцами `text` и `label`.

In [ ]:
%%writefile common.py
"""Общие функции для практикумов по классификации и кластеризации текстов."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "спорт": ("матч команда гол игрок тренер чемпионат турнир победа поражение сезон стадион болельщик судья счёт "
              "защитник нападающий вратарь кубок лига финал трансфер клуб удар пенальти соревнование спортсмен "
              "рекорд медаль олимпиада тренировка ничья").split(),
    "политика": ("закон депутат выборы партия голосование президент министр правительство парламент реформа бюджет "
                 "санкции кандидат коалиция министерство указ поправка законопроект оппозиция референдум дипломатия "
                 "мэр губернатор конституция заседание комиссия переговоры саммит избиратель мандат полномочия").split(),
    "кулинария": ("суп рецепт соль мясо готовить духовка овощи сковорода кастрюля тесто сахар специи варить жарить "
                  "запекать соус бульон салат повар блюдо ингредиент масло мука яйцо молоко чеснок лук перец "
                  "картофель десерт выпечка начинка маринад").split(),
    "технологии": ("процессор программа приложение смартфон нейросеть алгоритм данные сервер обновление разработчик "
                   "компьютер память экран камера браузер сеть интернет облако код платформа пользователь устройство "
                   "чип батарея ноутбук софт видеокарта система интерфейс шифрование").split(),
}
NEUTRAL = ("сегодня очень новый говорят после который также можно было стал сообщили вчера недавно ещё всё "
           "несколько важный первый").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """Учебный корпус из коротких 'документов' четырёх тем.
    noise: доля слов, взятых не из своей темы (нейтральные или чужие). Чем больше noise, тем труднее задача."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """Проекция разреженной матрицы TF-IDF на плоскость (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Печатает слова с наибольшими весами для каждого класса (линейные модели)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))

In [ ]:
from common import *

# Учебный корпус: 4 темы. noise - доля "посторонних" слов (чем больше, тем труднее).
df = make_corpus(n_per_class=150, noise=0.3, seed=0)
# Свой корпус вместо учебного: df = pd.read_csv("texts.csv")  # столбцы: text, label
print(df.label.value_counts().to_dict())
df.head()

## 1. Multiclass: один класс, вероятности в сумме дают 1

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, hamming_loss, f1_score, classification_report

vec = TfidfVectorizer()
Xa = vec.fit_transform(df.text)
mc = LogisticRegression(max_iter=1000).fit(Xa, df.label)
q = vec.transform(["матч команда гол бюджет закон депутат"])
p = mc.predict_proba(q)[0]
print(dict(zip(mc.classes_, p.round(3))), "сумма =", p.sum().round(3))

## 2. Multilabel: у документа может быть несколько меток
Создаём документы на стыке тем: склеиваем два текста разных тем, метки обе.

In [ ]:
rng = np.random.default_rng(0)
labs = sorted(df.label.unique())
pool = {l: df[df.label == l].text.tolist() for l in labs}
rows = []
for _ in range(300):
    a, b = rng.choice(labs, 2, replace=False)
    rows.append((rng.choice(pool[a]) + " " + rng.choice(pool[b]), [a, b]))
multi = pd.concat([pd.DataFrame({"text": df.text, "labels": [[l] for l in df.label]}),
                   pd.DataFrame(rows, columns=["text", "labels"])], ignore_index=True)
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(multi.labels)
Xm = vec.fit_transform(multi.text)
Xtr, Xte, Ytr, Yte = train_test_split(Xm, Y, test_size=0.3, random_state=0)
ovr = OneVsRestClassifier(LogisticRegression(C=10, max_iter=1000)).fit(Xtr, Ytr)
P = ovr.predict_proba(Xte)
print("Классы:", mlb.classes_)
print("Пример вероятностей (не в сумме 1):", P[0].round(2), "сумма =", P[0].sum().round(2))

In [ ]:
pred = (P >= 0.5).astype(int)
print("Точное совпадение набора меток:", round(accuracy_score(Yte, pred), 3))
print("Hamming loss:", round(hamming_loss(Yte, pred), 3))
print("F1 micro:", round(f1_score(Yte, pred, average="micro"), 3))
print(classification_report(Yte, pred, target_names=mlb.classes_, zero_division=0))

In [ ]:
ths = np.linspace(0.05, 0.95, 19)
plt.plot(ths, [f1_score(Yte, (P >= t).astype(int), average="micro") for t in ths], marker="o")
plt.xlabel("порог метки"); plt.ylabel("F1 micro"); plt.title("Выбор порога для multilabel"); plt.show()

## Задания

1. Подберите оптимальный порог отдельно для каждой метки. Что даёт индивидуальный порог?
2. Сравните `OneVsRestClassifier` с `LogisticRegression` (multinomial) на multiclass-версии тех же данных (оставьте одну метку на документ).
3. Приведите свой пример задачи multilabel из компьютерной лингвистики и предложите метрику для неё.